## Notes by Andrea

### Download data

In [20]:
import pandas as pd
import numpy as np
from pooch import retrieve

# Read data, put in pandas dataframe
url = "https://osf.io/download/qsb3f"
path = retrieve(url=url, known_hash=None)
data = pd.read_csv(path, encoding="latin1", sep = ';')
print(data.head())

               Source        AuthorName  Experiment  Year        Group  \
0  Goldstein.2008.Ex1  Goldstein et al.           1  2008      Control   
1  Goldstein.2008.Ex1  Goldstein et al.           1  2008  Social Norm   
2  Goldstein.2008.Ex1  Goldstein et al.           1  2008      Control   
3  Goldstein.2008.Ex1  Goldstein et al.           1  2008  Social Norm   
4  Goldstein.2008.Ex2  Goldstein et al.           2  2008      Control   

  Towel.Reuse  Count  
0         Yes     74  
1         Yes     98  
2          No    137  
3          No    124  
4         Yes    103  


### Data wrangling
Copied from exercise instructions

In [ ]:
count = data.iloc[:, -1] # get the last column with numbers or yes/no

# count has the number of yes and no for control and social norm groups
control_yes = count[::4].to_numpy() # every 4th starting from 0 - control group + yes
control_no = count[2::4].to_numpy() # every 4th starting from 2 - control group + no
control_total = np.array([y + n for y, n in zip(control_yes, control_no)])

social_yes = count[1::4].to_numpy() # every 4th starting from 1 - social norm group + yes
social_no = count[3::4].to_numpy() # every 4th starting from 3 - social norm group + no
social_total = np.array([y + n for y, n in zip(social_yes, social_no)])

study = np.arange(1,len(control_yes)+1) # 7 different studies
control_data = pd.DataFrame({"reuse": control_yes, "total": control_total, "group": "control", "study": study})
social_data = pd.DataFrame({ "reuse": social_yes, "total": social_total, "group": "social", "study": study})

combined_data = pd.concat([control_data, social_data], ignore_index=True)

# Convert data types - important for bambi that they are correctly set. an give errors otherwise
combined_data['reuse'] = combined_data['reuse'].astype(int)
combined_data['total'] = combined_data['total'].astype(int)
combined_data['group'] = combined_data['group'].astype('category')
combined_data['study'] = combined_data['study'].astype('category')



### Model specifications/Theory

The response variable follows a binomial distrubution

Use a model with a group specicig intercept for each study (intercept is a random effect). Use a shared slope (fixed effect). Means that the effect is overall the same (difference between no social norm and with social norm) but that the studies may differ in other ways - like choice of hotel or area, or time of the study.

Model to use (written using two different notations): 
$$
y_{ij} = \beta_0 + u_{j} + \beta_1 x_{ij} + \epsilon_{ij} \\
Y_{ij}|x \sim N(\beta_0 + u_{j} + \beta_1 x_{ij}, \sigma) 
$$
where $u_{ij}$ represents the random effect, $\beta_0$ represents the intercept and $\beta_1$ represents the slope. 
$$
u_{j} \sim N(0, \tau)
$$
The parameters are: $\beta_0, \beta_1, \sigma$ and $\tau$.
Hyper parameters are the following:
$$
\begin{aligned}
\beta_0 &\sim N(\mu_{\beta_0}, \sigma_{\beta_0}) \\
\beta_1 &\sim N(\mu_{\beta_1}, \sigma_{\beta_1}) \\
u_{j} &\sim \Gamma(a_\sigma, b_\sigma) \\
u_{j} &\sim \Gamma(a_\tau, b_\tau) 
\end{aligned}
$$